# 🧠 スキル自動進化ダッシュボード

v18 スキル自動進化システムの観察・候補・承認状況を管理します。

**テーブル:**
- `skill_observations` — 会話から検出されたパターン観察
- `skill_candidates` — 進化スキル候補（pending / approved / dismissed）
- `notion_skills` — Notion同期済みスキル

In [ ]:
import utils
from utils import qdf, scalar, ROLE_JA, PALETTE
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

## 1. 概要カウント

In [ ]:
obs_count   = scalar("SELECT COUNT(*) FROM skill_observations")
pending     = scalar("SELECT COUNT(*) FROM skill_candidates WHERE status='pending'")
approved    = scalar("SELECT COUNT(*) FROM skill_candidates WHERE status='approved'")
dismissed   = scalar("SELECT COUNT(*) FROM skill_candidates WHERE status='dismissed'")
notion_cnt  = scalar("SELECT COUNT(*) FROM notion_skills WHERE source='evolved'")

print('=' * 40)
print('  スキル進化システム 概要')
print('=' * 40)
print(f'  観察ログ          : {obs_count:>5}')
print(f'  スキル候補 (承認待): {pending:>5}')
print(f'  スキル候補 (承認済): {approved:>5}')
print(f'  スキル候補 (却下)  : {dismissed:>5}')
print(f'  Notion 同期済み   : {notion_cnt:>5}')
print('=' * 40)

## 2. スキル観察ログ

In [ ]:
obs = qdf("""
    SELECT role_used, keywords, pattern_hash,
           ROUND(execution_time::numeric, 0) AS ms,
           created_at
    FROM skill_observations
    ORDER BY created_at DESC
    LIMIT 50
""")

if utils.check_data(obs, 'スキル観察データ'):
    obs['役職'] = obs['role_used'].map(lambda r: ROLE_JA.get(r, r))
    display(obs[['役職','keywords','ms','created_at']].rename(
        columns={'keywords':'キーワード','ms':'実行時間(ms)','created_at':'日時'}
    ).set_index('日時'))

    # 役職別観察件数
    fig, ax = plt.subplots(figsize=(8, 4))
    obs['役職'].value_counts().plot(kind='barh', ax=ax, color=PALETTE[0], edgecolor='white')
    ax.set_title('役職別 スキル観察件数')
    ax.set_xlabel('件数')
    plt.tight_layout()
    plt.show()

## 3. スキル候補一覧

In [ ]:
candidates = qdf("""
    SELECT id, skill_name, keywords, typical_role, status,
           occurrence_count, proposed_at, approved_at
    FROM skill_candidates
    ORDER BY occurrence_count DESC, proposed_at DESC
""")

if utils.check_data(candidates, 'スキル候補'):
    candidates['役職'] = candidates['typical_role'].map(lambda r: ROLE_JA.get(r, r) if r else '')
    display(candidates[['skill_name','役職','keywords','status','occurrence_count','proposed_at']].rename(
        columns={'skill_name':'スキル名','keywords':'キーワード',
                 'status':'状態','occurrence_count':'出現回数','proposed_at':'提案日時'}
    ).set_index('スキル名'))

    if len(candidates['status'].unique()) > 1:
        fig, ax = plt.subplots(figsize=(6, 4))
        status_map = {'pending':'承認待ち', 'approved':'承認済み', 'dismissed':'却下'}
        status_counts = candidates['status'].map(status_map).value_counts()
        colors = {'承認待ち': PALETTE[4], '承認済み': PALETTE[2], '却下': '#AAAAAA'}
        status_counts.plot(kind='pie', ax=ax,
            autopct='%1.0f%%',
            colors=[colors.get(s, PALETTE[0]) for s in status_counts.index])
        ax.set_title('スキル候補 ステータス分布')
        ax.set_ylabel('')
        plt.tight_layout()
        plt.show()
else:
    print('ℹ️  スキル候補は /api/v18/evolve を実行すると自動生成されます')
    print('    （同じパターンのリクエストが一定回数蓄積されると候補になります）')

## 4. 承認待ちスキル — アクション一覧

In [ ]:
pending_df = qdf("""
    SELECT id, skill_name, keywords, typical_role, occurrence_count, sample_messages
    FROM skill_candidates
    WHERE status = 'pending'
    ORDER BY occurrence_count DESC
""")

if utils.check_data(pending_df, '承認待ちスキル'):
    print(f'承認待ち: {len(pending_df)} 件\n')
    for _, row in pending_df.iterrows():
        print(f'━━━ {row["skill_name"]} ━━━')
        print(f'  ID         : {row["id"]}')
        print(f'  役職       : {ROLE_JA.get(row["typical_role"], row["typical_role"])}')
        print(f'  キーワード : {row["keywords"]}')
        print(f'  出現回数   : {row["occurrence_count"]}')
        print('  操作ヒント : Web UI → システム設定 → スキル提案 で承認/却下')
        print()
else:
    print('✅ 承認待ちスキルはありません')

## 5. キーワード頻度分析

In [ ]:
kw_df = qdf("""
    SELECT keywords, COUNT(*) AS cnt
    FROM skill_observations
    WHERE keywords IS NOT NULL AND keywords != ''
    GROUP BY keywords
    ORDER BY cnt DESC
    LIMIT 20
""")

if utils.check_data(kw_df, 'キーワードデータ'):
    fig, ax = plt.subplots(figsize=(9, 5))
    kw_df.set_index('keywords')['cnt'].plot(
        kind='barh', ax=ax, color=PALETTE[1], edgecolor='white')
    ax.set_title('頻出キーワード TOP20')
    ax.set_xlabel('観察回数')
    plt.tight_layout()
    plt.show()